# SSG municipal income line items, 2025 — extract

Reads the BEP income sheet that SSG supplied inside `Analisis_Version_2.xlsx` and exports one row per
comuna and budget code. Nothing is classified or summed here: which codes count as own income,
other permanent income or FCM is a model decision and lives in the consuming release.

- Source workbook (gitignored): `sample/Analisis_Version_2.xlsx`, sheet `BEP INGRESOS MUNICIPALES 2025`
- Release contract: `review.md`

In [1]:
from pathlib import Path
import openpyxl
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "dataset-review").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / "dataset-review").exists(), "run from inside the CityCatalyst-global-data repository"

HERE = ROOT / "dataset-review/reviews/cl-ssg/cl-ssg-municipal-income/releases/2025"
SHEET = "BEP INGRESOS MUNICIPALES 2025"

wb = openpyxl.load_workbook(HERE / "sample/Analisis_Version_2.xlsx", read_only=True, data_only=True)
rows = list(wb[SHEET].iter_rows(values_only=True))
print(f"{len(rows)} sheet rows")

369 sheet rows


## Read the header block

Row 4 holds the budget codes and row 5 their labels. Data rows carry a comuna code in column B;
the interleaved region banner rows do not, and are skipped. The column labelled `Total` is the workbook's own
total, kept only for the check below.

In [2]:
codes = [str(c).strip() if c is not None else None for c in rows[3]]
labels = [str(l).strip() if l is not None else None for l in rows[4]]
assert labels[1:5] == ["Código", "Comuna", "Sector", "Trimestre"], labels[1:5]
assert labels.count("Total") == 1

code_cols = [i for i, c in enumerate(codes) if c and c.startswith("115.")]
total_col = labels.index("Total")
data = [r for r in rows[5:] if r[1] is not None and r[2] is not None]
print(f"{len(code_cols)} budget codes | {len(data)} comuna rows")

182 budget codes | 345 comuna rows


## Validate the sheet

One annual (fourth-quarter, cumulative) municipal-sector row per comuna, 345 comunas, integer
thousands of pesos. Each comuna's subtitle lines (`115.xx.00.000.000.000`) must add up to the
workbook's `Total`.

In [3]:
assert len(data) == 345
assert {str(r[3]).strip() for r in data} == {"MUNICIPAL"}
assert {r[4] for r in data} == {4}

subtitles = [i for i in code_cols if codes[i].split(".")[2] == "00"]
for r in data:
    assert all(isinstance(r[i], int) for i in code_cols + [total_col]), r[2]
    assert sum(r[i] for i in subtitles) == r[total_col], r[2]

wide = pd.DataFrame(
    [[str(r[1]).zfill(5), str(r[2]).strip()] + [r[i] for i in code_cols] for r in data],
    columns=["comuna_cut", "comuna"] + [codes[i] for i in code_cols])
assert wide["comuna_cut"].is_unique and wide["comuna_cut"].str.fullmatch(r"\d{5}").all()

blank = wide.loc[wide[[codes[i] for i in code_cols]].eq(0).all(axis=1), "comuna"].tolist()
print("comunas reporting nothing:", blank)

comunas reporting nothing: ['SAN PEDRO DE ATACAMA', 'RINCONADA', 'LOTA', 'CURACAUTÍN', 'FRUTILLAR']


## Export

One row per comuna, one column per budget code, as in the sheet. Values are integer thousands of
pesos on the cash-received basis. `reported` is false for the comunas whose row is entirely zero:
they reported nothing in this snapshot, which is not the same as zero income. Code labels are in
the source workbook and, for the codes the model uses, in the consuming classification table.

In [4]:
amount_cols = [codes[i] for i in code_cols]
wide.insert(2, "reported", ~wide["comuna"].isin(blank))
out = wide.sort_values("comuna_cut").reset_index(drop=True)

out.to_csv(HERE / "data/bep_income_lines_2025.csv", index=False)
print(f"wrote data/bep_income_lines_2025.csv — {out.shape[0]} comunas x {len(amount_cols)} budget codes, "
      f"{int(out['reported'].sum())} reporting")

wrote data/bep_income_lines_2025.csv — 345 comunas x 182 budget codes, 340 reporting


## Result

345 comunas are in the sheet and 340 report income; the five that report nothing are listed above.
The basis is cash received (*percibido*): the totals match SUBDERE's SIM/BEP percibido totals
wherever the two snapshots agree (see `review.md`).